# Grafico 03 - Ranking de areas protegidas por cambio en superficie natural

Este notebook reproduce en Google Colab las 2 imagenes de este grafico (ranking general y ranking por tipologia) y su tabla de soporte en Excel.

**Antes de correr las celdas de abajo**, ten a mano el archivo `naturalidad_data.json` (esta en la carpeta `codigo/` de este grafico, en tu computador o en tu repositorio de GitHub).

Corre las celdas en orden, de arriba hacia abajo.

## 1. Instalar paquetes

In [ ]:
!pip -q install numpy pandas matplotlib openpyxl


## 2. Subir el archivo de datos

In [ ]:
from google.colab import files
print("Sube aqui: naturalidad_data.json")
uploaded = files.upload()


## 3. Generar las 2 imagenes

In [ ]:
"""
Ranking de áreas protegidas por cambio en superficie natural — general + por tipología
==========================================================================================

Gráfico corregido según comentarios de Vale (agosto-2026). Ver README.txt y
METODOLOGIA.docx de esta carpeta para el detalle completo de qué cambió y
por qué.

QUÉ HACE ESTE SCRIPT
--------------------
Genera 2 gráficos de barras horizontales con las AP de mayor cambio (en
puntos porcentuales) en su % de superficie natural interna entre 2000 y
2024:

  1. ranking_general.png            -> top 10 con mayor PÉRDIDA + top 10
                                        con mayor GANANCIA, entre las 97 AP
                                        juntas (20 barras en total). Antes
                                        este gráfico mostraba 8+8; Vale
                                        pidió ampliarlo a 10+10 (\"top 10\").
  2. ranking_por_tipologia.png      -> lo mismo, pero calculado POR
                                        SEPARADO dentro de cada tipología
                                        (Parques / Reservas / Monumentos),
                                        en 3 paneles -- para comparar cuáles
                                        son las AP más extremas DENTRO de
                                        su propia categoría, no solo en el
                                        ranking nacional. Acá se usa top 5
                                        + 5 por panel (en vez de 10+10)
                                        porque Monumentos Naturales solo
                                        tiene 14 AP en total -- con 10+10
                                        casi no quedaría ninguna AP fuera
                                        del gráfico, y no sería comparable
                                        en proporción con los paneles de
                                        Parques (42 AP) y Reservas (41 AP).

DISEÑO: solo título + nombres de ejes + el gráfico (más la leyenda de
color Pérdida/Ganancia, que es parte del gráfico, no una nota aparte) --
sin subtítulo ni notas al pie sobre la imagen (esa explicación va en el
README/METODOLOGIA). Terminología corregida: \"superficie natural\", no
\"cobertura natural\".

ARCHIVO DE ENTRADA (debe estar en esta misma carpeta `codigo/`)
------------------------------------------------------------------
  naturalidad_data.json   -> % de superficie natural por AP, año y
                              distancia (acá se usa la distancia \"AP\", es
                              decir el % natural dentro del polígono de la
                              AP misma, comparando 2000 vs. 2024).

SALIDA (se guarda en ../imagenes/)
-----------------------------------
  ranking_general.png          -> top 10 pérdidas + top 10 ganancias (97 AP)
  ranking_por_tipologia.png    -> top 5 pérdidas + top 5 ganancias, calculado
                                   por separado dentro de cada tipología

Para correrlo: python3 ranking.py
(requiere numpy, matplotlib -- instalar con: pip install numpy matplotlib)

===========================================================================
QUÉ CAMBIAR SI...                                                (resumen)
===========================================================================
  ...moviste este script a otra carpeta y naturalidad_data.json no está al
     lado -> variable NATURALIDAD_JSON_PATH, más abajo.
  ...quieres que las imágenes se guarden en otro lugar
     -> variable OUT_DIR, más abajo.
  ...quieres cambiar cuántas AP se muestran en cada extremo
     -> variables N_TOP_GENERAL (hoy 10) y N_TOP_TIPOLOGIA (hoy 5), más
        abajo.
  ...cambian los años que se comparan (hoy: 2000 vs 2024)
     -> buscar \"2000\" y \"2024\" dentro de calcular_cambios().
  ...quieres cambiar tamaño de letra, colores, tamaño de figura, etc.
     (ajustes puramente visuales)
     -> están marcados con \"<-- AJUSTE VISUAL\" en cada sección.
===========================================================================
"""

import json
import re
import os
import numpy as np
import matplotlib
matplotlib.use(\"Agg\")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ---------------------------------------------------------------------
# RUTAS DE ARCHIVOS
# ---------------------------------------------------------------------
BASE_DIR = \"/content\"  # <-- en Colab, los archivos subidos con files.upload() quedan en /content

# <-- CAMBIAR AQUÍ si le cambiaste el nombre al archivo de datos, o si lo
#     moviste a otra carpeta.
NATURALIDAD_JSON_PATH = os.path.join(BASE_DIR, \"naturalidad_data.json\")

# <-- CAMBIAR AQUÍ si quieres que los PNG se guarden en otro lugar (por
#     defecto: una carpeta \"imagenes\" al lado de esta carpeta \"codigo\").
OUT_DIR = os.path.join(BASE_DIR, \"imagenes\")
os.makedirs(OUT_DIR, exist_ok=True)

# <-- CAMBIAR AQUÍ cuántas AP se muestran en cada extremo (pérdida/ganancia).
N_TOP_GENERAL = 10    # ranking_general.png: top N pérdidas + top N ganancias, sobre las 97 AP
N_TOP_TIPOLOGIA = 5   # ranking_por_tipologia.png: top N pérdidas + top N ganancias, DENTRO de cada tipología

# ------------------------------------------------------------------
# PALETA Y ESTILO (igual al resto del proyecto, para consistencia visual)
# ------------------------------------------------------------------
SURFACE = \"#fcfcfb\"
INK_PRIMARY = \"#0b0b0b\"
INK_SECONDARY = \"#52514e\"
INK_MUTED = \"#898781\"
GRID = \"#e1e0d9\"
BASELINE = \"#c3c2b7\"
DIV_BLUE = \"#2a78d6\"   # azul = ganancia de superficie natural
DIV_RED = \"#e34948\"    # rojo = pérdida de superficie natural

plt.rcParams[\"font.family\"] = \"sans-serif\"
plt.rcParams[\"font.sans-serif\"] = [\"DejaVu Sans\", \"Arial\", \"Helvetica\"]


def style_ax(ax):
    ax.set_facecolor(SURFACE)
    for s in [\"top\", \"right\"]:
        ax.spines[s].set_visible(False)
    for s in [\"left\", \"bottom\"]:
        ax.spines[s].set_color(BASELINE)
    ax.tick_params(colors=INK_MUTED, labelsize=9)
    ax.xaxis.label.set_color(INK_SECONDARY)
    ax.yaxis.label.set_color(INK_SECONDARY)


# ------------------------------------------------------------------
# 1. CARGA DE DATOS
# ------------------------------------------------------------------
DATA = json.load(open(NATURALIDAD_JSON_PATH))
DIST = DATA[\"dist\"]
APS = DATA[\"aps\"]
ANILLOS = DATA[\"anillos\"]


def tipologia(nombre):
    \"\"\"PN/RN/MN a partir del prefijo del nombre de la AP.
    <-- CAMBIAR AQUÍ si el formato de los nombres cambia en el futuro.\"\"\"
    m = re.match(r\"^(MN|PN|RN)\\s\", nombre)
    return m.group(1) if m else \"??\"


TIPO_ORDER = [\"PN\", \"RN\", \"MN\"]  # <-- CAMBIAR AQUÍ si se agrega una 4ta tipología
TIPO_NOMBRE = {\"PN\": \"Parque Nacional\", \"RN\": \"Reserva Nacional\", \"MN\": \"Monumento Natural\"}

for a in APS:
    a[\"tipo\"] = tipologia(a[\"name\"])


def ap_val(ap_name, year, dist_label):
    idx = DIST.index(dist_label)
    return ANILLOS[ap_name][str(year)][idx]


def calcular_cambios(subset):
    \"\"\"Devuelve [(nombre_AP, cambio_en_puntos_porcentuales), ...] para las
    AP de `subset`, ordenado de mayor pérdida a mayor ganancia.
    <-- CAMBIAR AQUÍ los años 2000/2024 si se quiere comparar otro par.\"\"\"
    out = []
    for a in subset:
        v0 = ap_val(a[\"name\"], 2000, \"AP\")
        v1 = ap_val(a[\"name\"], 2024, \"AP\")
        if v0 is None or v1 is None:
            continue
        out.append((a[\"name\"], v1 - v0))
    out.sort(key=lambda r: r[1])
    return out


# ------------------------------------------------------------------
# 2. FUNCIÓN QUE DIBUJA UN PANEL DE RANKING (barras horizontales)
#    dentro de un eje (ax) ya existente -- se reutiliza tanto para el
#    gráfico general (1 panel) como para el de por tipología (3 paneles).
# ------------------------------------------------------------------
def dibujar_ranking(ax, cambios, n_top, mostrar_ylabel_eje=True):
    losers = cambios[:n_top]
    gainers = cambios[-n_top:] if n_top <= len(cambios) else cambios[len(cambios) // 2:]
    top = losers + gainers

    style_ax(ax)
    ypos = np.arange(len(top))
    vals = [r[1] for r in top]
    colors = [DIV_RED if v < 0 else DIV_BLUE for v in vals]
    ax.barh(ypos, vals, color=colors, height=0.62, zorder=3)  # <-- AJUSTE VISUAL: height=0.62 (grosor de barra)
    ax.axvline(0, color=BASELINE, linewidth=1, zorder=2)
    for y, (name, v) in zip(ypos, top):
        label_x = v - 0.6 if v < 0 else v + 0.6
        ha = \"right\" if v < 0 else \"left\"
        ax.text(label_x, y, name, va=\"center\", ha=ha, fontsize=8.3, color=INK_SECONDARY)
        # <-- AJUSTE VISUAL: fontsize=8.3 (tamaño del nombre de la AP); 0.6 (separación con la barra)
    ax.set_yticks([])
    ax.axhline(len(losers) - 0.5, color=GRID, linewidth=0.8, zorder=1)  # línea separadora pérdidas/ganancias
    if mostrar_ylabel_eje:
        ax.set_xlabel(\"Cambio en % superficie natural (2024 − 2000, puntos porcentuales)\")
    ax.grid(axis=\"x\", color=GRID, linewidth=0.7, zorder=0)
    ax.set_axisbelow(True)
    margen = max(4, (max(vals) - min(vals)) * 0.22)  # <-- AJUSTE VISUAL: margen proporcional para que quepan las etiquetas
    ax.set_xlim(min(vals) - margen, max(vals) + margen)
    return len(top)


# ------------------------------------------------------------------
# 3. RANKING GENERAL — top N_TOP_GENERAL pérdidas + top N_TOP_GENERAL ganancias, 97 AP
# ------------------------------------------------------------------
cambios_general = calcular_cambios(APS)
fig, ax = plt.subplots(figsize=(8.4, 8.6), dpi=200)  # <-- AJUSTE VISUAL: tamaño y resolución de la imagen
fig.patch.set_facecolor(SURFACE)
dibujar_ranking(ax, cambios_general, N_TOP_GENERAL)

handles = [Patch(facecolor=DIV_RED, label=\"Pérdida\"), Patch(facecolor=DIV_BLUE, label=\"Ganancia\")]
leg = ax.legend(handles=handles, loc=\"upper center\", bbox_to_anchor=(0.5, 1.045),
                 frameon=False, fontsize=9.5, ncol=2)
for t in leg.get_texts():
    t.set_color(INK_SECONDARY)

# Solo título -- sin subtítulo/nota metodológica debajo (esa explicación
# va en el README/METODOLOGIA de esta carpeta).
fig.suptitle(f\"Áreas protegidas con mayor cambio en la superficie\\nnatural, 2000–2024 (top {N_TOP_GENERAL} pérdidas y ganancias)\",
             color=INK_PRIMARY, fontsize=14, fontweight=\"bold\", x=0.02, ha=\"left\", y=0.995, va=\"top\")
             # <-- AJUSTE VISUAL: fontsize=14 y posición (x, y) del título
fig.subplots_adjust(top=0.85, bottom=0.07, left=0.03, right=0.97)
fig.savefig(os.path.join(OUT_DIR, \"ranking_general.png\"), facecolor=SURFACE)
plt.close(fig)
print(f\"OK ranking_general.png (top {N_TOP_GENERAL}+{N_TOP_GENERAL}, {len(cambios_general)} AP con dato completo)\")

# ------------------------------------------------------------------
# 4. RANKING POR TIPOLOGÍA — 3 paneles, top N_TOP_TIPOLOGIA dentro de
#    cada tipología (no dentro del ranking nacional completo)
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 7.4), dpi=200)  # <-- AJUSTE VISUAL: tamaño de la imagen
fig.patch.set_facecolor(SURFACE)
for i, (ax, t) in enumerate(zip(axes, TIPO_ORDER)):
    aps_t = [a for a in APS if a[\"tipo\"] == t]
    cambios_t = calcular_cambios(aps_t)
    n_mostradas = dibujar_ranking(ax, cambios_t, N_TOP_TIPOLOGIA, mostrar_ylabel_eje=False)
    ax.set_title(f\"{TIPO_NOMBRE[t]} ({len(aps_t)} AP en total)\", color=INK_PRIMARY,
                 fontsize=11, fontweight=\"bold\", pad=10)
axes[1].set_xlabel(\"Cambio en % superficie natural (2024 − 2000, puntos porcentuales)\")
# <-- AJUSTE VISUAL: la etiqueta del eje X solo se pone en el panel del
#     medio (Reservas) para no repetirla 3 veces; si se prefiere repetirla
#     en los 3 paneles, quitar el \"if\" de mostrar_ylabel_eje en dibujar_ranking()

fig.suptitle(f\"Áreas protegidas con mayor cambio en la superficie natural, 2000–2024\\npor tipología (top {N_TOP_TIPOLOGIA} pérdidas y ganancias dentro de cada categoría)\",
             color=INK_PRIMARY, fontsize=14, fontweight=\"bold\", x=0.02, ha=\"left\", y=0.998, va=\"top\")
             # <-- AJUSTE VISUAL: fontsize=14 y posición (x, y) del título (2 líneas)

# La leyenda se pone DEBAJO del título de 2 líneas (y=0.855, no 0.975)
# para que no se superpongan -- en una iteración anterior quedaban
# encimados porque el título ocupa más alto de lo que parece a primera
# vista al tener 2 líneas de texto.
handles = [Patch(facecolor=DIV_RED, label=\"Pérdida\"), Patch(facecolor=DIV_BLUE, label=\"Ganancia\")]
leg = fig.legend(handles=handles, loc=\"upper center\", bbox_to_anchor=(0.5, 0.855),
                  frameon=False, fontsize=9.5, ncol=2)  # <-- AJUSTE VISUAL: posición y tamaño de la leyenda
for t_ in leg.get_texts():
    t_.set_color(INK_SECONDARY)

fig.subplots_adjust(top=0.72, bottom=0.07, left=0.03, right=0.97, wspace=0.28)
# <-- AJUSTE VISUAL: top=0.72 deja espacio para el título (2 líneas) + la
#     leyenda encima de los paneles; si se agranda el título hay que subir
#     este margen también.
fig.savefig(os.path.join(OUT_DIR, \"ranking_por_tipologia.png\"), facecolor=SURFACE)
plt.close(fig)
print(f\"OK ranking_por_tipologia.png (top {N_TOP_TIPOLOGIA}+{N_TOP_TIPOLOGIA} por tipología)\")


## 4. Generar la tabla de soporte (Excel)

In [ ]:
"""
Tabla de soporte del gráfico 04 (ranking de AP por cambio en superficie natural)
====================================================================================

Genera tabla_soporte.xlsx (en la carpeta de arriba, junto a README.txt) con
los datos exactos que usan los 2 gráficos de ranking, en 2 hojas:

  1. ranking_general    -> las 97 AP, ordenadas de mayor pérdida a mayor
                            ganancia (no solo el top 10+10 que se ve en la
                            imagen -- acá está el ranking completo).
  2. ranking_por_tipologia -> lo mismo, pero calculado dentro de cada
                            tipología por separado (columna \"tipologia\"),
                            con el orden dentro de su propia categoría.

Requiere: numpy, pandas, openpyxl
Para correrlo: python3 tabla_soporte.py

===========================================================================
QUÉ CAMBIAR SI...                                                (resumen)
===========================================================================
  ...moviste este script a otra carpeta y naturalidad_data.json no está al
     lado -> variable NATURALIDAD_JSON_PATH, más abajo.
  ...quieres que tabla_soporte.xlsx se guarde en otro lugar
     -> variable OUT_XLSX, más abajo.
===========================================================================
"""

import json
import re
import os
import pandas as pd

BASE_DIR = \"/content\"  # <-- en Colab, los archivos subidos con files.upload() quedan en /content
NATURALIDAD_JSON_PATH = os.path.join(BASE_DIR, \"naturalidad_data.json\")
OUT_XLSX = os.path.join(BASE_DIR, \"tabla_soporte.xlsx\")

DATA = json.load(open(NATURALIDAD_JSON_PATH))
DIST = DATA[\"dist\"]
APS = DATA[\"aps\"]
ANILLOS = DATA[\"anillos\"]


def tipologia(nombre):
    m = re.match(r\"^(MN|PN|RN)\\s\", nombre)
    return m.group(1) if m else \"??\"


TIPO_NOMBRE = {\"PN\": \"Parque Nacional\", \"RN\": \"Reserva Nacional\", \"MN\": \"Monumento Natural\"}

for a in APS:
    a[\"tipo\"] = tipologia(a[\"name\"])


def ap_val(ap_name, year, dist_label):
    idx = DIST.index(dist_label)
    return ANILLOS[ap_name][str(year)][idx]


# ---------------------------------------------------------------
# 1) HOJA ranking_general: las 97 AP ordenadas de mayor pérdida a
#    mayor ganancia (2000 vs 2024, % natural dentro de la AP)
# ---------------------------------------------------------------
rows = []
for a in APS:
    v0 = ap_val(a[\"name\"], 2000, \"AP\")
    v1 = ap_val(a[\"name\"], 2024, \"AP\")
    if v0 is None or v1 is None:
        continue
    rows.append({
        \"AP\": a[\"name\"], \"tipologia\": a[\"tipo\"], \"macrozona\": a[\"macro\"],
        \"pct_natural_2000\": v0, \"pct_natural_2024\": v1,
        \"cambio_puntos_porcentuales\": round(v1 - v0, 2),
    })
df_rank = pd.DataFrame(rows).sort_values(\"cambio_puntos_porcentuales\").reset_index(drop=True)
df_rank.insert(0, \"posicion\", range(1, len(df_rank) + 1))

# ---------------------------------------------------------------
# 2) HOJA ranking_por_tipologia: mismo cálculo, pero con la posición
#    (ranking) calculada DENTRO de cada tipología por separado.
# ---------------------------------------------------------------
df_tipo = df_rank.drop(columns=\"posicion\").copy()
df_tipo[\"posicion_dentro_de_tipologia\"] = (
    df_tipo.groupby(\"tipologia\")[\"cambio_puntos_porcentuales\"].rank(method=\"first\").astype(int)
)
df_tipo = df_tipo.sort_values([\"tipologia\", \"posicion_dentro_de_tipologia\"]).reset_index(drop=True)

with pd.ExcelWriter(OUT_XLSX, engine=\"openpyxl\") as writer:
    df_rank.to_excel(writer, sheet_name=\"ranking_general\", index=False)
    df_tipo.to_excel(writer, sheet_name=\"ranking_por_tipologia\", index=False)

print(f\"OK {OUT_XLSX} -- {len(df_rank)} AP, 2 hojas\")


## 5. Ver las imagenes generadas

In [ ]:
import glob
from IPython.display import Image, display

for p in sorted(glob.glob(os.path.join(OUT_DIR, '*.png'))):
    print(p.split('/')[-1])
    display(Image(filename=p))


## 6. Descargar todo (imagenes + tabla de soporte) en un .zip

In [ ]:
import shutil, os
from google.colab import files

RESULT_DIR = \"/content/resultados_03_ranking\"
os.makedirs(RESULT_DIR, exist_ok=True)
if os.path.isdir(OUT_DIR):
    shutil.copytree(OUT_DIR, os.path.join(RESULT_DIR, \"imagenes\"), dirs_exist_ok=True)
if os.path.exists(OUT_XLSX):
    shutil.copy(OUT_XLSX, RESULT_DIR)
shutil.make_archive(RESULT_DIR, \"zip\", RESULT_DIR)
files.download(RESULT_DIR + \".zip\")
print(\"Listo:\", RESULT_DIR + \".zip\")
